# ✂️ Notebook 3: Text Chunking
### Why we split text into pieces — and how

**WAT Reference:** `workflows/03_chunk_text.md` | `tools/chunk_text.py`

---

## The Problem with Long Documents

Your MBA case studies can be 20, 50, or 100 pages long. GPT-4 can't read all of it at once — there's a **context window limit** (the maximum amount of text an LLM can process in one go).

But more importantly: even if GPT-4 *could* read 100 pages, sending 100 pages for every question would be:
- **Slow** (processing time scales with length)
- **Expensive** ($0.50+ per question instead of $0.001)
- **Imprecise** (GPT-4 works better with focused context)

The solution: split each document into **small, focused pieces** and only send the 5 most relevant pieces per question.

## What is a Chunk?

```
CAPM and Portfolio Theory.pdf  (50 pages, ~25,000 words)
  ↓  chunking
Chunk 0: "The Capital Asset Pricing Model (CAPM) describes..."
Chunk 1: "...relationship between systematic risk and expected return.
          The formula is: E(Ri) = Rf + βi(E(Rm) − Rf) where..."
Chunk 2: "...Beta (β) measures a stock's volatility relative to the market.
          A beta of 1.0 means the stock moves with the market..."
            :
Chunk 49: "...In conclusion, CAPM provides a theoretical framework..."
```

50 pages → ~50 chunks, each ~500 words. When you ask about CAPM, we retrieve only the 5 most relevant chunks.

## What You'll Do

1. Understand tokens vs words
2. See why we use overlap between chunks
3. Build the chunker from scratch
4. Visualize overlapping chunks side-by-side
5. Chunk all sample docs → save to `.tmp/chunks.json`

**Time needed:** ~15 minutes | **Cost:** $0 (no API calls)

In [ ]:
#@title ⚙️ Step 1 — Install tokenizer library  { display-mode: "form" }
!pip install tiktoken -q
print("✅ tiktoken installed — OpenAI's tokenizer")

In [ ]:
#@title 📁 Step 2 — Connect to Google Drive  { display-mode: "form" }
import os, json

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except Exception:
    IN_COLAB = False
    print("ℹ️  Google Drive is not available in this environment.")
    print("   To run this notebook with your MBA files, open it in Google Colab.")

PROJECT_ROOT = '/content/drive/MyDrive/MBA_RAG'
INPUT_FILE   = os.path.join(PROJECT_ROOT, '.tmp/parsed_docs.json')
OUTPUT_FILE  = os.path.join(PROJECT_ROOT, '.tmp/chunks.json')

if IN_COLAB and os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        documents = json.load(f)
    print(f"✅ Loaded {len(documents)} documents from parsed_docs.json")
    for doc in documents:
        print(f"   {doc['filename']:<45} ({doc['subject']})")
elif IN_COLAB:
    print(f"⚠️  parsed_docs.json not found. Run Notebook 2 first.")
    documents = []
else:
    documents = []

---
## Step 1: Tokens — Not Words, Not Characters

We measure chunk size in **tokens**, not words or characters. Why?

OpenAI's API charges and enforces limits by tokens. A token is roughly:
- 1 common word = 1 token (`the`, `is`, `market`)
- Long or rare words = 2-3 tokens (`competitive`, `profitability`)
- Numbers = varies (`2024` = 1 token, `$1,234,567` = several tokens)

**Rule of thumb:** 500 tokens ≈ 375 words ≈ 1.5 pages

In [ ]:
#@title 🔢 Step 3 — See how text becomes tokens  { display-mode: "form" }
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

sample_texts = [
    "Porter's Five Forces",
    "The Capital Asset Pricing Model",
    "Competitive advantage arises from unique positioning",
    "EBITDA = Earnings Before Interest, Taxes, Depreciation and Amortisation",
]

print("TEXT → TOKENS DEMO")
print("=" * 70)
print(f"{'Text':<55} {'Tokens':>6} {'Words':>6}")
print("─" * 70)

for text in sample_texts:
    tokens = tokenizer.encode(text)
    words  = len(text.split())
    print(f"{text:<55} {len(tokens):>6} {words:>6}")

print()
print("💡 Rule of thumb: 500 tokens ≈ 375 words ≈ 1.5 pages")
print()
print("Token counts for your documents:")
print("─" * 60)
for doc in documents:
    token_count = len(tokenizer.encode(doc['text']))
    chunks_est  = max(1, token_count // 500)
    print(f"  {doc['filename']:<40} {token_count:>6} tokens → ~{chunks_est} chunks")

---
## Step 2: Why Overlap? The Boundary Problem

Without overlap, cutting at exactly 500 tokens can split a sentence or idea in half:

```
Without overlap:                      |  With 50-token overlap:
                                       |
Chunk 0: "...CAPM formula is          |  Chunk 0: "...CAPM formula is
          E(Ri) = Rf + β(Rm-Rf)        |            E(Ri) = Rf + β(Rm-Rf)
          where Rf is the risk-        |            where Rf is the risk-
          free rate and"               |            free rate and"           ← last 50 tokens
                                       |                                       repeated
Chunk 1: "β represents systematic     |  Chunk 1: "free rate and             ← first 50 tokens
          risk relative to..."         |            β represents systematic    from chunk 0
                                       |            risk relative to..."
                                       |
Problem: 'and' is a dangling word.    |  Solution: context preserved
What does 'and' connect to? Lost!     |  across the boundary.
```

In [ ]:
#@title ✂️ Step 4 — Build the chunker (with overlap)  { display-mode: "form" }
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

def split_into_chunks(text, tokenizer, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    tokens = tokenizer.encode(text)
    if len(tokens) <= chunk_size:
        return [tokenizer.decode(tokens)]
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunks.append(tokenizer.decode(tokens[start:end]))
        start += (chunk_size - overlap)
    return chunks

if documents:
    doc    = documents[0]
    chunks = split_into_chunks(doc['text'], tokenizer)
    print(f"Document: {doc['filename']}")
    print(f"Total tokens: {len(tokenizer.encode(doc['text'])):,}")
    print(f"Chunks created: {len(chunks)}")
    if len(chunks) >= 2:
        print()
        print("OVERLAP VISUALIZATION — end of Chunk 0 connects to start of Chunk 1:")
        print("─" * 60)
        print(f'End of Chunk 0:   ...{chunks[0][-150:].strip()}')
        print()
        print(f'Start of Chunk 1: {chunks[1][:150].strip()}...')
        print("─" * 60)
        print()
        print("The shared text ensures context is never lost at boundaries.")
else:
    print("Load documents first (Step 2).")

---
## Step 3: Chunk All Documents → Save to `.tmp/chunks.json`

In [ ]:
#@title 🔄 Step 5 — Chunk ALL documents and save  { display-mode: "form" }
all_chunks = []
chunk_id   = 0

print("Chunking all documents...")
print()

for doc_num, doc in enumerate(documents, 1):
    chunks_list = split_into_chunks(doc['text'], tokenizer)
    token_count = len(tokenizer.encode(doc['text']))
    print(f"[{doc_num}/{len(documents)}] {doc['filename']}")
    print(f"  Subject: {doc['subject']} | Tokens: {token_count:,} → {len(chunks_list)} chunk(s)")
    for chunk_index, chunk_text in enumerate(chunks_list):
        all_chunks.append({
            'chunk_id':    chunk_id,
            'chunk_index': chunk_index,
            'text':        chunk_text,
            'token_count': len(tokenizer.encode(chunk_text)),
            'filename':    doc['filename'],
            'source':      doc['source'],
            'subject':     doc['subject'],
        })
        chunk_id += 1

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

total_tokens = sum(c['token_count'] for c in all_chunks)
print()
print("=" * 60)
print(f"✅ Total chunks created: {len(all_chunks):,}")
print(f"   Total tokens:         {total_tokens:,}")
print(f"   Avg tokens/chunk:     {total_tokens // len(all_chunks) if all_chunks else 0}")
print(f"   Saved to:             {OUTPUT_FILE}")

In [ ]:
#@title 🔍 Step 6 — Inspect chunk structure  { display-mode: "form" }
print("SAMPLE CHUNK STRUCTURE")
print("=" * 60)

first_doc_chunks = [c for c in all_chunks if c['chunk_index'] <= 1][:2]
for chunk in first_doc_chunks:
    print(f"\nchunk_id:    {chunk['chunk_id']}")
    print(f"chunk_index: {chunk['chunk_index']}  (position within document)")
    print(f"filename:    {chunk['filename']}")
    print(f"subject:     {chunk['subject']}  (inherited from folder)")
    print(f"token_count: {chunk['token_count']}")
    print(f"text:        {chunk['text'][:200]}...")

print()
print("💡 In Notebook 4, each chunk gets one more field: 'embedding' (1,536 numbers).")

---
## ✅ Notebook 3 Complete!

### What you learned:
1. **Tokens** = the unit OpenAI uses to measure text length (~1 token per word)
2. **Chunking** = splitting long documents into focused 500-token pieces
3. **Overlap** = sharing 50 tokens between adjacent chunks to preserve context at boundaries
4. **Subject tag** = inherited from the parent document's folder name
5. Output: `.tmp/chunks.json` — all chunks with metadata, ready for embedding

### The data flow so far:
```
data/sample/*.pdf/.docx/.pptx/.xlsx
         ↓  Notebook 2 (parse)
.tmp/parsed_docs.json   [{filename, subject, text}, ...]
         ↓  Notebook 3 (chunk)
.tmp/chunks.json        [{chunk_id, text, subject, token_count, ...}, ...]
         ↓  Notebook 4 (embed)
.tmp/vector_store.json  [{chunk_id, text, subject, embedding: [1536 numbers]}, ...]
```

---
### Next: Notebook 4 — Embeddings

This is the most important concept in RAG. We convert each chunk's text into **1,536 numbers** that capture its meaning. Similar texts get similar numbers. This is what makes semantic search possible.

**⚠️ Cost warning:** Notebook 4 will call the OpenAI API and cost ~$0.001–$0.01 for your sample files. There's a cost estimator before any charges.

**Before starting Notebook 4:**
1. Open `.tmp/chunks.json` in Google Drive — verify it looks right
2. Count the chunks — multiply by 500 tokens, divide by 1M, multiply by $0.02 = your cost
3. Update README.md — check off Notebook 3 ✅